# 15. Recuperación densa automatizada

Procesa los **10 modelos pendientes** y agrega sus resultados a `densa_resultados_v1.json`. No recalcula embeddings de artículos ni modifica los índices. **BGE-M3 se conserva.**

Ejecutar desde `07_RAG/Pruebas_RAG/` con el mismo entorno CUDA usado en los notebooks de los modelos. El modelo Qwen3-0.6B necesita verificar su revisión local: si no puede identificarse sin ambigüedad, se omite y queda documentado.

### 1. Entradas y configuración

In [1]:
import gc, hashlib, json, platform, sys
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import faiss

REPETICIONES = 10
SHA_CATALOGO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
SHA_CONSULTAS = "1e35b789207b49ebb3e1e088c342cda234c2153f7d1b50f0d05adbf8cf459f17"
MODELOS = ["granite_311m_r2", "harrier_0_6b", "nemotron3_1b", "arctic_l_v2",
           "embeddinggemma2_text", "pplx_0_6b", "qwen3_vl_2b_text",
           "qwen3_4b", "pplx_4b_bf16", "qwen3_0_6b"]

def sha_archivo(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def sha_json(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, ensure_ascii=False,
        separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/consultas_evaluacion_v1.csv").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre VS Code dentro de Tesis_Multimodelo.")
RAG = REPO / "07_RAG"
PRUEBAS = RAG / "Pruebas_RAG"
SALIDA = PRUEBAS / "densa_resultados_v1.json"
BM25 = PRUEBAS / "bm25_resultados_v1.json"
C = REPO / "06_grafos/catalogo_articulos.csv"
Q = RAG / "consultas_evaluacion_v1.csv"
assert sha_archivo(C) == SHA_CATALOGO and sha_archivo(Q) == SHA_CONSULTAS
assert SALIDA.is_file() and BM25.is_file(), "Primero conserva los resultados originales de BGE-M3 y BM25."

catalogo = pd.read_csv(C, dtype=str, keep_default_na=False)
consultas_df = pd.read_csv(Q, dtype=str, keep_default_na=False)
ids = catalogo["article_id"].tolist()
ids_num = np.array([int(x) for x in ids], dtype=np.int64)
consultas = consultas_df[["query_id", "necesidad_id", "division", "idioma", "consulta"]].to_dict("records")
assert len(ids) == 519 and len(set(ids)) == 519 and len(consultas) == 24

registro = json.loads(SALIDA.read_text(encoding="utf-8"))
assert registro["version_esquema"] == "RECUPERACION_DENSA_V1"
assert registro["fuentes"] == {"sha256_catalogo": SHA_CATALOGO, "sha256_consultas": SHA_CONSULTAS}
assert registro["consultas"] == consultas and "bge_m3" in registro["sistemas"]
for sistema in registro["sistemas"].values():
    assert sha_json(sistema["resultados"]) == sistema["sha256_resultados"]

bm25 = json.loads(BM25.read_text(encoding="utf-8"))
assert bm25["fuentes"]["catalogo"]["sha256"] == SHA_CATALOGO
assert bm25["fuentes"]["consultas"]["sha256"] == SHA_CONSULTAS
ref_bm25 = {"ruta": "07_RAG/Pruebas_RAG/bm25_resultados_v1.json",
            "sha256_archivo_al_inicio": sha_archivo(BM25),
            "sha256_resultados": bm25["sha256_resultados"],
            "utilizado_para_puntuar": False}
del bm25

LIBRETA = PRUEBAS / "15_recuperacion_densa_automatizada.ipynb"
if not LIBRETA.is_file():
    raise FileNotFoundError("Guarda esta libreta en Pruebas_RAG con su nombre original.")
notebook = json.loads(LIBRETA.read_text(encoding="utf-8"))
sha_codigo = sha_json(["".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"])
faiss.omp_set_num_threads(1)
print("BGE-M3 registrado | 10 pendientes como máximo | 24 consultas | 519 artículos")

BGE-M3 registrado | 10 pendientes como máximo | 24 consultas | 519 artículos


### 2. Verificación y codificación por modelo

In [2]:
import torch
import transformers, sentence_transformers, huggingface_hub
from sentence_transformers import SentenceTransformer

torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
if not torch.cuda.is_available():
    raise RuntimeError("Se requiere el entorno CUDA usado para generar los embeddings.")

VERSIONES = {"torch": torch.__version__, "transformers": transformers.__version__,
            "sentence_transformers": sentence_transformers.__version__}
ORIGINAL = registro["sistemas"]["bge_m3"]["ejecuciones"][0]["entorno"]
for biblioteca, version in VERSIONES.items():
    if version != ORIGINAL[biblioteca]:
        raise RuntimeError(f"Versión diferente de {biblioteca}: {version}, antes {ORIGINAL[biblioteca]}")

def revision_qwen_06():
    repos = [r for r in huggingface_hub.scan_cache_dir().repos
             if r.repo_id == "Qwen/Qwen3-Embedding-0.6B" and r.repo_type == "model"]
    revs = sorted({v.commit_hash for r in repos for v in r.revisions})
    if len(revs) != 1:
        raise ValueError("Qwen3-0.6B: la revisión no quedó registrada y la caché no contiene un único snapshot; se omite.")
    return revs[0]

def datos_modelo(clave):
    p_npz = RAG / f"{clave}_vectores_v1.npz"
    p_idx = RAG / f"{clave}_indice_v1.faiss"
    with np.load(p_npz, allow_pickle=False) as z:
        assert z["article_ids"].tolist() == ids
        assert z["sha256_catalogo"].item() == SHA_CATALOGO
        assert z["campo"].item() == "abstract"
        model_id = z["modelo"].item()
        rev = z["modelo_revision"].item()
        matriz = z["vectores"]
        assert matriz.shape[0] == 519 and matriz.dtype == np.float32
        assert np.isfinite(matriz).all()
        assert np.allclose(np.linalg.norm(matriz, axis=1), 1, rtol=0, atol=1e-5)
        if "metadata_json" in z:
            meta = json.loads(z["metadata_json"].item())
            perfil = meta["perfil"]
            assert meta["estado"] == "completo" and perfil["modelo"] == model_id
            assert perfil["revision"] == rev
            assert int(perfil["dimensiones"]) == matriz.shape[1]
            assert hashlib.sha256(matriz.tobytes()).hexdigest() == z["sha256_matriz"].item()
            if meta.get("sha256_indice"):
                assert sha_archivo(p_idx) == meta["sha256_indice"]
        else:
            perfil = {"modelo": model_id, "revision": rev,
                      "dimensiones": int(matriz.shape[1]),
                      "prefijo_consulta": z["prefijo_consulta"].item() if "prefijo_consulta" in z else "",
                      "limite_tokens": int(z["max_seq_length"].item()) if "max_seq_length" in z else None}
        if clave == "qwen3_0_6b":
            assert rev == "no_registrada" and z["dtype_modelo"].item() == "torch.bfloat16"
        elif clave == "qwen3_4b":
            assert z["dtype_modelo"].item() == "torch.bfloat16"
        else:
            assert rev != "no_registrada"
        indice = faiss.read_index(str(p_idx))
        assert isinstance(indice, faiss.IndexFlatIP)
        assert indice.ntotal == 519 and indice.d == matriz.shape[1]
        assert np.array_equal(indice.reconstruct_n(0, 519), matriz)
    perfil["revision"] = rev
    perfil["dimensiones"] = int(indice.d)
    if clave == "qwen3_0_6b":
        perfil.update(revision=revision_qwen_06(), nombre_prompt_consulta="query",
                      precision_verificada="bfloat16", revision_procedencia="snapshot_unico_cache_inferido")
    if clave == "qwen3_4b":
        perfil.update(nombre_prompt_consulta="query", precision_verificada="bfloat16")
    return perfil, indice, {"npz": p_npz.name, "sha256_npz": sha_archivo(p_npz),
                            "indice": p_idx.name, "sha256_indice": sha_archivo(p_idx)}

def cargar_modelo(clave, p):
    dtype_str = (p.get("dtype_pesos") or p.get("dtype_pesos_solicitado") or
                 p.get("precision_verificada") or "bfloat16")
    dtype = torch.float32 if "float32" in dtype_str else torch.bfloat16
    kw = {"dtype": dtype}
    ck = {}
    if clave == "qwen3_4b":
        kw = {"torch_dtype": torch.bfloat16}
    elif clave == "qwen3_0_6b":
        kw = {"torch_dtype": torch.bfloat16}
    elif clave == "harrier_0_6b":
        pass
    elif clave == "granite_311m_r2":
        kw["attn_implementation"] = "sdpa"
        ck = {"reference_compile": False}
    elif clave == "embeddinggemma2_text":
        kw.update(attn_implementation="sdpa", use_safetensors=True)
        ck = {"vision_config": None, "audio_config": None}
    elif clave in ("arctic_l_v2", "qwen3_vl_2b_text", "pplx_4b_bf16"):
        kw.update(attn_implementation="sdpa", use_safetensors=True)
    elif clave in ("nemotron3_1b", "pplx_0_6b"):
        kw["attn_implementation"] = "sdpa"
    if clave.startswith("pplx_"):
        kw["code_revision"] = p["revision"]
        ck = {"code_revision": p["revision"]}
    args = {"revision": p["revision"], "device": "cuda",
            "trust_remote_code": clave.startswith("pplx_"), "model_kwargs": kw}
    if ck:
        args["config_kwargs"] = ck
    m = SentenceTransformer(p["modelo"], **args)
    m.eval()
    expected = int(p["dimensiones"])
    get_dim = getattr(m, "get_embedding_dimension", None)
    if not callable(get_dim):
        get_dim = m.get_sentence_embedding_dimension
    assert int(get_dim()) == expected
    original_limit = int(m.max_seq_length)
    if p.get("limite_tokens") is not None:
        m.max_seq_length = min(original_limit, int(p["limite_tokens"]))
    if p.get("max_seq_length") is not None:
        m.max_seq_length = min(int(m.max_seq_length), int(p["max_seq_length"]))
    if clave == "qwen3_4b":
        m.max_seq_length = min(original_limit, 32768)
    if clave == "qwen3_0_6b":
        assert m.prompts.get("query") and m.default_prompt_name is None
    else:
        name = p.get("nombre_prompt_consulta")
        pref = p.get("prefijo_consulta", "")
        if name:
            assert m.prompts.get(name) == pref, f"Prefijo incorrecto: {clave}"
            assert m.default_prompt_name == ("default" if clave == "qwen3_vl_2b_text" else None)
        else:
            assert m.default_prompt_name is None
            assert not any(x for x in m.prompts.values() if x not in (None, ""))
    if p.get("pooling"):
        pol = next((x for x in m if type(x).__name__ == "Pooling"), None)
        assert pol is not None
        conf = pol.get_config_dict()
        modo = conf.get("pooling_mode")
        if modo is None:
            modos = {"cls": "pooling_mode_cls_token", "mean": "pooling_mode_mean_tokens",
                     "lasttoken": "pooling_mode_lasttoken"}
            modo = next((x for x, bandera in modos.items() if conf.get(bandera)), None)
        assert modo == p["pooling"], f"Pooling distinto: {clave}"
    if clave.startswith("pplx_"):
        assert len(m) == 3 and type(m[2]).__name__ == "FlexibleQuantizer"
    if clave == "embeddinggemma2_text":
        config = m[0].auto_model.config
        assert getattr(config, "vision_config", None) is None and getattr(config, "audio_config", None) is None
    if clave == "qwen3_vl_2b_text":
        assert m[0].auto_model.config.model_type == "qwen3_vl"
    dtypes = set(str(v.dtype) for v in m.parameters())
    if clave in ("embeddinggemma2_text", "qwen3_vl_2b_text"):
        assert "torch.bfloat16" in dtypes and dtypes <= {"torch.bfloat16", "torch.float32"}
    else:
        assert dtypes == {str(dtype)}, f"Precisión inesperada: {dtypes}"
    assert all(v.device.type == "cuda" for v in m.parameters())
    config = getattr(getattr(m[0], "auto_model", None), "config", None)
    commit = getattr(config, "_commit_hash", None)
    if commit is not None:
        assert commit == p["revision"]
    return m, {"dtype_modelo": sorted(dtypes), "revision_solicitada": p["revision"],
               "revision_reportada": commit, "max_seq_length": int(m.max_seq_length),
               "prompt_name": p.get("nombre_prompt_consulta"),
               "prefijo_consulta": p.get("prefijo_consulta", ""),
               "revision_procedencia": p.get("revision_procedencia", "fijada_en_npz")}

def longitud_consulta(m, clave, p, texto):
    pref = p.get("prefijo_consulta", "")
    if clave == "qwen3_vl_2b_text":
        datos = m.preprocess([{"text": texto}], prompt=pref,
            processing_kwargs={"text": {"truncation": False, "padding": True},
                               "chat_template": {"add_generation_prompt": True}})
        assert all(datos.get(x) is None for x in ("pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw"))
        n = int(datos["attention_mask"].sum().item())
    elif clave == "embeddinggemma2_text":
        datos = m.preprocess([texto], prompt=pref,
            processing_kwargs={"text": {"truncation": False, "padding": True}})
        n = int(datos["attention_mask"].sum().item())
    else:
        tok = m.tokenizer([pref + texto], add_special_tokens=True, padding=False, truncation=False)
        n = len(tok["input_ids"][0])
    assert 0 < n <= int(m.max_seq_length), f"Consulta demasiado larga: {clave}"
    return n

def codificar(m, clave, texto):
    kwargs = {"batch_size": 1, "show_progress_bar": False, "convert_to_numpy": True}
    if clave in ("granite_311m_r2", "nemotron3_1b"):
        salida = m.encode_query([texto], normalize_embeddings=True, **kwargs)
    elif clave.startswith("pplx_"):
        salida = m.encode([texto], prompt="", normalize_embeddings=False,
                          precision="float32", quantization="int8", **kwargs)
        v = np.asarray(salida)
        assert float(v.min()) >= -128.001 and float(v.max()) <= 127.001
    elif clave == "qwen3_vl_2b_text":
        salida = m.encode([{"text": texto}], prompt_name="default", normalize_embeddings=True,
                          precision="float32", processing_kwargs={"text": {"truncation": False, "padding": True},
                          "chat_template": {"add_generation_prompt": True}}, **kwargs)
    elif clave == "embeddinggemma2_text":
        salida = m.encode([texto], prompt_name="query", normalize_embeddings=True,
                          precision="float32", processing_kwargs={"text": {"truncation": False, "padding": True}}, **kwargs)
    elif clave == "harrier_0_6b":
        salida = m.encode([texto], prompt_name="web_search_query", normalize_embeddings=True, **kwargs)
    elif clave in ("qwen3_0_6b", "qwen3_4b", "arctic_l_v2"):
        salida = m.encode([texto], prompt_name="query", normalize_embeddings=True, **kwargs)
    else:
        raise ValueError(clave)
    v = np.array(salida, dtype=np.float32, order="C", copy=True)
    assert v.ndim == 2 and v.shape[0] == 1 and np.isfinite(v).all()
    assert np.linalg.norm(v) > 0
    faiss.normalize_L2(v)
    assert np.allclose(np.linalg.norm(v, axis=1), 1, rtol=0, atol=1e-5)
    return v

W1108 16:17:37.898000 22940 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


### 3. Ejecutar y guardar cada modelo terminado

Un fallo no borra los modelos completados. Al repetir la libreta, se omiten automáticamente los sistemas ya guardados. La carga de los modelos PPLX ejecuta el código oficial de la revisión fijada, como en tus notebooks originales.

In [3]:
def estadistica(valores):
    return {"muestras_ms": valores, "mediana_ms": float(np.median(valores)),
            "minimo_ms": min(valores), "maximo_ms": max(valores)}

def guardar_seguro():
    global sha_anterior
    actual = sha_archivo(SALIDA)
    if actual != sha_anterior:
        raise RuntimeError("El JSON fue modificado externamente. No se sobrescribe.")
    tmp = SALIDA.with_suffix(".json.tmp")
    if tmp.exists():
        raise FileExistsError("Existe densa_resultados_v1.json.tmp; revisa la escritura interrumpida.")
    try:
        tmp.write_text(json.dumps(registro, ensure_ascii=False, separators=(",", ":"), allow_nan=False) + "\n", encoding="utf-8")
        tmp.replace(SALIDA)
    finally:
        if tmp.exists():
            tmp.unlink()
    sha_anterior = sha_archivo(SALIDA)

def ejecutar(clave):
    p, indice, archivos = datos_modelo(clave)
    if clave == "qwen3_0_6b":
        print("  Aviso: revisión de Qwen3-0.6B inferida de caché; pendiente de corroboración externa.")
    condicion = {
        "protocolo": "RECUPERACION_DENSA_V1", "modelo": p["modelo"], "revision": p["revision"],
        "revision_procedencia": p.get("revision_procedencia", "fijada_en_npz"),
        "dtype_pesos": p.get("dtype_pesos") or p.get("dtype_pesos_solicitado") or p.get("precision_verificada"),
        "dimension": int(p["dimensiones"]), "campo_documento": "abstract", "campo_consulta": "consulta",
        "prefijo_consulta": p.get("prefijo_consulta", ""),
        "nombre_prompt_consulta": p.get("nombre_prompt_consulta"),
        "batch_size_consultas": 1, "dispositivo_codificador": "cuda",
        "normalizacion": "float32 y L2", "indice": "IndexFlatIP CPU", "hilos_faiss": 1,
        "profundidad": 519, "empates": "score descendente, article_id ascendente",
        "calentamientos_por_consulta": 1, "repeticiones_por_consulta": REPETICIONES,
        "ranking_guardado": "primera medicion despues del calentamiento",
        "qrels_utilizados": False, "metricas_calidad_calculadas": False,
    }
    firma = sha_json({"condicion": condicion, "datos": registro["fuentes"], "archivos_modelo": archivos})
    if clave in registro["sistemas"]:
        assert registro["sistemas"][clave]["firma_condicion"] == firma
        print(f"  {clave}: ya existe; sin inferencias nuevas")
        return None
    m = None
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        libre_gpu, total_gpu = torch.cuda.mem_get_info()
        inicio_carga = perf_counter()
        m, runtime = cargar_modelo(clave, p)
        torch.cuda.synchronize()
        carga_ms = 1000 * (perf_counter() - inicio_carga)
        tokens = [longitud_consulta(m, clave, p, q["consulta"]) for q in consultas]
        resultados, latencias = [], []
        for n, q in enumerate(consultas):
            texto = q["consulta"]
            def buscar():
                torch.cuda.synchronize()
                t0 = perf_counter()
                with torch.inference_mode():
                    vec = codificar(m, clave, texto)
                torch.cuda.synchronize()
                t1 = perf_counter()
                scores, posiciones = indice.search(vec, 519)
                scores, posiciones = scores[0], posiciones[0]
                orden = np.lexsort((ids_num[posiciones], -scores))
                scores, posiciones = scores[orden], posiciones[orden]
                t2 = perf_counter()
                return vec, scores, posiciones, {"codificacion": (t1-t0)*1000,
                   "busqueda_orden": (t2-t1)*1000, "total": (t2-t0)*1000}
            buscar()
            tiempos = {k: [] for k in ("codificacion", "busqueda_orden", "total")}
            estable = True
            diferencia = 0.0
            for it in range(REPETICIONES):
                v, sc, pos, tt = buscar()
                for k in tiempos:
                    tiempos[k].append(tt[k])
                if it == 0:
                    v0, sc0, pos0 = v.copy(), sc.copy(), pos.copy()
                    base_scores = np.empty(519, dtype=np.float32)
                    base_scores[pos] = sc
                else:
                    otros_scores = np.empty(519, dtype=np.float32)
                    otros_scores[pos] = sc
                    diferencia = max(diferencia, float(np.max(np.abs(base_scores - otros_scores))))
                    estable &= bool(np.array_equal(pos0, pos))
            assert sorted(pos0.tolist()) == list(range(519))
            assert np.isfinite(sc0).all() and np.all(np.abs(sc0) <= 1.00001)
            resultados.append({"query_id": q["query_id"], "tokens_consulta": tokens[n],
                "vector_consulta": v0[0].tolist(),
                "documentos": [{"article_id": ids[int(i)], "score_coseno": float(score), "rank": rank}
                               for rank, (i, score) in enumerate(zip(pos0, sc0), start=1)]})
            latencias.append({"query_id": q["query_id"],
                **{k: estadistica(valores) for k, valores in tiempos.items()},
                "max_diferencia_coseno_repeticiones": diferencia,
                "mismo_orden_en_todas_repeticiones": estable})
        assert len(resultados) == 24
        entorno = {"python": sys.version.split()[0], "torch": torch.__version__,
            "sentence_transformers": sentence_transformers.__version__,
            "transformers": transformers.__version__, "numpy": np.__version__,
            "faiss": getattr(faiss, "__version__", "desconocida"),
            "gpu": torch.cuda.get_device_name(0), "sistema": platform.platform()}
        ejecucion = {"fin_utc": datetime.now(timezone.utc).isoformat(), "entorno": entorno,
            "sha256_codigo": sha_codigo, "carga_modelo_ms": carga_ms,
            "calentamientos_por_consulta": 1, "repeticiones_por_consulta": REPETICIONES,
            "latencias_consulta": latencias,
            "alcance_latencia": "consulta individual: encode + L2 float32 + FAISS519 + orden por ID",
            "gpu_total_bytes": int(total_gpu), "gpu_libre_antes_carga_bytes": int(libre_gpu),
            "pico_pytorch_asignado_bytes": int(torch.cuda.max_memory_allocated()),
            "pico_pytorch_reservado_bytes": int(torch.cuda.max_memory_reserved()),
            "ram_pico_bytes": None}
        return {"firma_condicion": firma, "condicion": condicion,
            "fuentes": archivos, "modelo_runtime": runtime,
            "referencia_bm25": ref_bm25, "resultados": resultados,
            "sha256_resultados": sha_json(resultados), "ejecuciones": [ejecucion]}
    finally:
        if m is not None:
            del m
        gc.collect()
        torch.cuda.empty_cache()

sha_anterior = sha_archivo(SALIDA)
for n, clave in enumerate(MODELOS, start=1):
    print(f"[{n}/{len(MODELOS)}] {clave}", flush=True)
    if clave in registro["sistemas"]:
        print("  Registrado anteriormente: omitido")
        continue
    try:
        resultado = ejecutar(clave)
        if resultado is not None:
            registro["sistemas"][clave] = resultado
            guardar_seguro()
            tiempos = [r["total"]["mediana_ms"] for r in resultado["ejecuciones"][0]["latencias_consulta"]]
            print(f"  Guardado: 24 consultas, 12 456 puntuaciones, mediana {np.median(tiempos):.2f} ms")
    except Exception as e:
        aviso = {"modelo": clave, "fecha_utc": datetime.now(timezone.utc).isoformat(),
                 "tipo": type(e).__name__, "detalle": str(e)[:500]}
        registro.setdefault("incidencias_automatizacion", []).append(aviso)
        print("  ERROR:", aviso["tipo"], aviso["detalle"])
        if sha_archivo(SALIDA) == sha_anterior:
            guardar_seguro()
        else:
            raise RuntimeError("El archivo de salida fue modificado externamente.") from e
print("\nFinalizado. Modelos guardados:", len(registro["sistemas"]), "/ 11")

[1/10] granite_311m_r2


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

  Guardado: 24 consultas, 12 456 puntuaciones, mediana 19.81 ms
[2/10] harrier_0_6b


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Guardado: 24 consultas, 12 456 puntuaciones, mediana 46.49 ms
[3/10] nemotron3_1b


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


... (14 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  Guardado: 24 consultas, 12 456 puntuaciones, mediana 60.13 ms
[10/10] qwen3_0_6b
  Aviso: revisión de Qwen3-0.6B inferida de caché; pendiente de corroboración externa.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Guardado: 24 consultas, 12 456 puntuaciones, mediana 45.96 ms

Finalizado. Modelos guardados: 11 / 11


### 4. Comprobación final

In [4]:
final = json.loads(SALIDA.read_text(encoding="utf-8"))
filas = []
for k, s in final["sistemas"].items():
    assert sha_json(s["resultados"]) == s["sha256_resultados"]
    assert len(s["resultados"]) == 24
    assert all(len(q["documentos"]) == 519 for q in s["resultados"])
    lat = s["ejecuciones"][-1]["latencias_consulta"]
    filas.append({"modelo": k, "consultas": len(s["resultados"]),
                  "mediana_total_ms": round(float(np.median([x["total"]["mediana_ms"] for x in lat])), 2),
                  "revision": s["condicion"].get("revision_procedencia", "fijada_en_npz")})
display(pd.DataFrame(filas))
print("Incidencias:", len(final.get("incidencias_automatizacion", [])))
print("Archivo:", SALIDA)
print("No se calcularon métricas de pertinencia.")

,modelo,consultas,mediana_total_ms,revision
0,bge_m3,24,20.59,fijada_en_npz
1,granite_311m_r2,24,19.81,fijada_en_npz
2,harrier_0_6b,24,46.49,fijada_en_npz
3,nemotron3_1b,24,24.13,fijada_en_npz
4,arctic_l_v2,24,20.02,fijada_en_npz
5,embeddinggemma2_text,24,56.75,fijada_en_npz
6,pplx_0_6b,24,44.43,fijada_en_npz
7,qwen3_vl_2b_text,24,48.36,fijada_en_npz
8,qwen3_4b,24,61.71,fijada_en_npz
9,pplx_4b_bf16,24,60.13,fijada_en_npz


Incidencias: 0
Archivo: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\densa_resultados_v1.json
No se calcularon métricas de pertinencia.
